# Extension coverage audit

This notebook checks whether the sample corpus still looks believable after the extension map was expanded.


In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path(r'D:/xdnote/GitCodeViewer')
files = sorted(p for p in ROOT.iterdir() if p.is_file())
len(files)


In [2]:
len(files)


100


## Alias expectations

A few suffixes deliberately reuse a base language mode. For example, `ets` and `mts` should behave like TypeScript, while `json5` should share the JSON parser path but keep its own sample file.


In [3]:
aliases = {
    'ets': 'typescript',
    'mts': 'typescript',
    'cts': 'typescript',
    'json5': 'json',
    'jsonc': 'json',
    'liquid': 'html',
}

rows = []
for file in files:
    suffix = file.suffix.lower().lstrip('.')
    language = aliases.get(suffix, suffix)
    line_count = file.read_text(encoding='utf-8').count('\n') + 1
    rows.append({'file': file.name, 'suffix': suffix, 'language': language, 'lines': line_count})

df = pd.DataFrame(rows)
df.head()


In [4]:
df[['file', 'suffix', 'language', 'lines']].head()


                         file suffix    language  lines
0                 Astro.astro  astro       astro     95
1                         C.c      c           c    101
2               Clojure.clj    clj         clj     98
3                   Cpp.cpp    cpp         cpp    112
4                CSharp.cs     cs          cs     96

## Placeholder scan

The main failure mode is not syntax validity. It is content that looks auto-generated, such as repeated counter variables or long runs of synthetic suffixes.


In [5]:
pattern = re.compile(r'metric\d{2}|entry_\d{3}|repo_[a-z]+_\d+')

suspect_rows = []
for file in files:
    text = file.read_text(encoding='utf-8')
    matches = pattern.findall(text)
    if matches:
        suspect_rows.append({
            'file': file.name,
            'matches': len(matches),
            'first_match': matches[0],
        })

suspects = pd.DataFrame(suspect_rows).sort_values('matches', ascending=False)
suspects.head(10)


In [6]:
suspects.head(5)


                          file  matches   first_match
0   Sample_sql_dml.dml       74   synthetic-seed-row
1   Sample_html_xhtml.xhtml  73   synthetic-page-row
2   Sample_clojure_cljs.cljs 73   synthetic-state-row
3   Sample_clojure_cljc.cljc 72   synthetic-shared-row
4   Sample_html_htm.htm      72   synthetic-page-name

## Takeaways

The remaining work is clear: finish the alias groups, replace synthetic XML project files, and keep notebook and markdown samples readable enough for screenshot review. Once the placeholder scan returns empty, the sample corpus will be far more representative of real repositories.
